In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.express as px

pd.set_option("display.max_columns", 50)

df = pd.read_csv(
    "globalterrorismdb_0718dist.tar.bz2",
    compression="bz2",
    encoding="ISO-8859-1",
    low_memory=False,
)
print(df.shape)
df.head()

(181691, 136)


,Unnamed: 0,eventid,iyear,imonth,iday,approxdate,extended,resolution,country,country_txt,region,region_txt,provstate,city,latitude,longitude,specificity,vicinity,location,summary,crit1,crit2,crit3,doubtterr,alternative,...,nhostkid,nhostkidus,nhours,ndays,divert,kidhijcountry,ransom,ransomamt,ransomamtus,ransompaid,ransompaidus,ransomnote,hostkidoutcome,hostkidoutcome_txt,nreleased,addnotes,scite1,scite2,scite3,dbsource,INT_LOG,INT_IDEO,INT_MISC,INT_ANY,related
0,0,197000000001,1970,7,2,NaN,0,NaN,58,Dominican Republic,2,Central America & Caribbean,NaN,Santo Domingo,18.456792,-69.951164,1.0,0,NaN,NaN,1,1,1,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PGIS,0,0,0,0,NaN
1,1,197000000002,1970,0,0,NaN,0,NaN,130,Mexico,1,North America,Federal,Mexico city,19.371887,-99.086624,1.0,0,NaN,NaN,1,1,1,0.0,NaN,...,1.0,0.0,NaN,NaN,NaN,Mexico,1.0,800000.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PGIS,0,1,1,1,NaN
2,2,197001000001,1970,1,0,NaN,0,NaN,160,Philippines,5,Southeast Asia,Tarlac,Unknown,15.478598,120.599741,4.0,0,NaN,NaN,1,1,1,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PGIS,-9,-9,1,1,NaN
3,3,197001000002,1970,1,0,NaN,0,NaN,78,Greece,8,Western Europe,Attica,Athens,37.997490,23.762728,1.0,0,NaN,NaN,1,1,1,0.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PGIS,-9,-9,1,1,NaN
4,4,197001000003,1970,1,0,NaN,0,NaN,101,Japan,4,East Asia,Fukouka,Fukouka,33.580412,130.396361,1.0,0,NaN,NaN,1,1,1,-9.0,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,0.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,PGIS,-9,-9,1,1,NaN


In [3]:
print(df.columns.tolist())
df.isna().mean().sort_values(ascending=False).head(20)

['Unnamed: 0', 'eventid', 'iyear', 'imonth', 'iday', 'approxdate', 'extended', 'resolution', 'country', 'country_txt', 'region', 'region_txt', 'provstate', 'city', 'latitude', 'longitude', 'specificity', 'vicinity', 'location', 'summary', 'crit1', 'crit2', 'crit3', 'doubtterr', 'alternative', 'alternative_txt', 'multiple', 'success', 'suicide', 'attacktype1', 'attacktype1_txt', 'attacktype2', 'attacktype2_txt', 'attacktype3', 'attacktype3_txt', 'targtype1', 'targtype1_txt', 'targsubtype1', 'targsubtype1_txt', 'corp1', 'target1', 'natlty1', 'natlty1_txt', 'targtype2', 'targtype2_txt', 'targsubtype2', 'targsubtype2_txt', 'corp2', 'target2', 'natlty2', 'natlty2_txt', 'targtype3', 'targtype3_txt', 'targsubtype3', 'targsubtype3_txt', 'corp3', 'target3', 'natlty3', 'natlty3_txt', 'gname', 'gsubname', 'gname2', 'gsubname2', 'gname3', 'gsubname3', 'motive', 'guncertain1', 'guncertain2', 'guncertain3', 'individual', 'nperps', 'nperpcap', 'claimed', 'claimmode', 'claimmode_txt', 'claim2', 'claim

gsubname3           0.999890
weapsubtype4        0.999615
weapsubtype4_txt    0.999615
weaptype4_txt       0.999598
weaptype4           0.999598
claimmode3          0.999268
claimmode3_txt      0.999268
gsubname2           0.999119
claim3              0.998250
guncertain3         0.998239
divert              0.998217
gname3              0.998217
attacktype3         0.997644
attacktype3_txt     0.997644
ransomnote          0.997182
ransompaidus        0.996962
ransomamtus         0.996901
claimmode2          0.996610
claimmode2_txt      0.996610
ransompaid          0.995740
dtype: float64

In [4]:
cols = [
    "eventid", "iyear", "imonth", "iday",
    "country_txt", "region_txt", "city", "latitude", "longitude",
    "success", "suicide",
    "attacktype1_txt", "targtype1_txt", "targsubtype1_txt", "target1", "natlty1_txt",
    "gname", "nperps",
    "weaptype1_txt", "weapsubtype1_txt",
    "nkill", "nwound", "nkillus",
]

# keep only the columns that exist in this extract
cols = [c for c in cols if c in df.columns]
data = df[cols].copy()
print(data.shape)

(181691, 23)


In [5]:
# 1. Unknown-value codes: GTD uses -99 / -9 for "unknown" in numeric fields
data["nperps"] = data["nperps"].replace([-99, -9], np.nan)

# 2. Casualties: combine killed + wounded, treating missing as unknown (not zero)
data["casualties"] = data[["nkill", "nwound"]].sum(axis=1, min_count=1)

# 3. Coordinates: keep a separate frame for maps
geo = data.dropna(subset=["latitude", "longitude"])
print(f"Incidents with coordinates: {len(geo):,} of {len(data):,}")

# 4. Quick sanity checks
data[["iyear", "nkill", "nwound", "casualties"]].describe()

Incidents with coordinates: 177,134 of 181,691


,iyear,nkill,nwound,casualties
count,181691.000000,171378.000000,165380.000000,171941.000000
mean,2002.638997,2.403272,3.167668,5.442198
std,13.259430,11.545741,35.949392,41.670360
min,1970.000000,0.000000,0.000000,0.000000
25%,1991.000000,0.000000,0.000000,0.000000
50%,2009.000000,0.000000,0.000000,1.000000
75%,2014.000000,2.000000,2.000000,4.000000
max,2017.000000,1570.000000,8191.000000,9574.000000


In [6]:
data.nlargest(10, "casualties")[["iyear", "country_txt", "city", "gname", "attacktype1_txt", "nkill", "nwound", "casualties"]]

,iyear,country_txt,city,gname,attacktype1_txt,nkill,nwound,casualties
73126,2001,United States,New York City,Al-Qaida,Hijacking,1384.0,8190.0,9574.0
73127,2001,United States,New York City,Al-Qaida,Hijacking,1383.0,8191.0,9574.0
58841,1995,Japan,Tokyo,Aum Shinri Kyo,Unarmed Assault,13.0,5500.0,5513.0
68071,1998,Kenya,Nairobi,Al-Qaida,Bombing/Explosion,224.0,4000.0,4224.0
133518,2014,Iraq,Tikrit,Islamic State of Iraq and the Levant (ISIL),Hostage Taking (Kidnapping),1570.0,NaN,1570.0
159773,2016,Iraq,Taza Khurmatu,Islamic State of Iraq and the Levant (ISIL),Bombing/Explosion,3.0,1500.0,1503.0
61548,1996,Sri Lanka,Colombo,Liberation Tigers of Tamil Eelam (LTTE),Bombing/Explosion,90.0,1272.0,1362.0
55934,1994,Rwanda,Gikoro,Hutu extremists,Armed Assault,1180.0,0.0,1180.0
85682,2008,Chad,N'Djamena,Rebels,Armed Assault,160.0,1001.0,1161.0
76953,2004,Russia,Beslan,Riyadus-Salikhin Reconnaissance and Sabotage B...,Hostage Taking (Barricade Incident),344.0,727.0,1071.0


In [7]:
country_year = (
    data.groupby(["country_txt", "iyear"])
        .agg(incidents=("eventid", "size"),
             killed=("nkill", "sum"),
             wounded=("nwound", "sum"))
        .reset_index()
)
country_year["casualties"] = country_year["killed"] + country_year["wounded"]

rho = country_year[["incidents", "casualties"]].corr(method="spearman").iloc[0, 1]
print(f"Spearman correlation (country-year): {rho:.2f}")

px.scatter(country_year, x="incidents", y="casualties",
           log_x=True, log_y=True,
           hover_data=["country_txt", "iyear"],
           title="Incidents vs casualties by country-year (log-log)")

Spearman correlation (country-year): 0.76


In [8]:
country_year["cas_per_incident"] = country_year["casualties"] / country_year["incidents"]
country_year[country_year["incidents"] >= 3].nlargest(10, "cas_per_incident")

,country_txt,iyear,incidents,killed,wounded,casualties,cas_per_incident
3502,United States,2001,41,3008.0,16515.0,19523.0,476.170732
1751,Japan,1995,20,15.0,6225.0,6240.0,312.000000
608,Chad,2008,8,172.0,1011.0,1183.0,147.875000
551,Canada,1985,3,331.0,2.0,333.0,111.000000
1526,Iraq,1976,3,12.0,285.0,297.0,99.000000
287,Bangladesh,2002,5,21.0,440.0,461.0,92.200000
87,Angola,1983,7,282.0,349.0,631.0,90.142857
3359,Uganda,1984,3,235.0,0.0,235.0,78.333333
109,Angola,2017,6,7.0,411.0,418.0,69.666667
2208,Mozambique,1986,7,470.0,17.0,487.0,69.571429


In [9]:
yearly = data.groupby("iyear").size().rename("incidents").reset_index()

fig = px.line(yearly, x="iyear", y="incidents", markers=True,
              title="Terrorist incidents per year (1970-2017)",
              labels={"iyear": "Year", "incidents": "Incidents"})
fig.add_vline(x=1993, line_dash="dot", line_color="gray",
              annotation_text="1993: records incomplete")
fig.show()

In [11]:
by_region = (data.groupby(["iyear", "region_txt"]).size()
                 .rename("incidents").reset_index())

px.line(by_region, x="iyear", y="incidents", color="region_txt",
        title="Incidents per year by region",
        labels={"iyear": "Year", "incidents": "Incidents", "region_txt": "Region"}).show()

# ---------- 3. Share of global incidents by region ----------
by_region["share"] = (by_region["incidents"]
                      / by_region.groupby("iyear")["incidents"].transform("sum"))

px.area(by_region, x="iyear", y="share", color="region_txt",
        title="Share of global incidents by region",
        labels={"iyear": "Year", "share": "Share of incidents", "region_txt": "Region"}).show()

# ---------- 4. Trend indexed to each region's own peak ----------
pivot = by_region.pivot(index="iyear", columns="region_txt", values="incidents").fillna(0)
global_series = pivot.sum(axis=1)

indexed = pivot.div(pivot.max())
indexed["Global"] = global_series / global_series.max()

px.line(indexed.reset_index().melt(id_vars="iyear", var_name="region", value_name="index"),
        x="iyear", y="index", color="region",
        title="Incident trend indexed to each region's peak (1.0 = peak year)",
        labels={"iyear": "Year", "index": "Index (peak = 1)"}).show()

# ---------- 5. Peak year per region ----------
peaks = (pivot.idxmax().rename("peak_year").to_frame()
              .join(pivot.max().rename("peak_incidents"))
              .sort_values("peak_incidents", ascending=False))
display(peaks)

,peak_year,peak_incidents
region_txt,,
Middle East & North Africa,2014,6939.0
South Asia,2014,4998.0
Sub-Saharan Africa,2014,2321.0
South America,1984,1492.0
Southeast Asia,2013,1188.0
Central America & Caribbean,1981,1148.0
Western Europe,1979,1020.0
Eastern Europe,2014,962.0
North America,1970,472.0


In [12]:
from IPython.display import display

# ---------- 1. Share chart with distinct colors ----------
fig = px.area(by_region, x="iyear", y="share", color="region_txt",
              color_discrete_sequence=px.colors.qualitative.Dark24,
              title="Share of global incidents by region",
              labels={"iyear": "Year", "share": "Share of incidents", "region_txt": "Region"})
fig.update_layout(hovermode="x unified", height=550)
fig.show()

# ---------- 2. Regional share of incidents by decade (%) ----------
by_region["decade"] = (by_region["iyear"] // 10) * 10
dec = by_region.groupby(["decade", "region_txt"])["incidents"].sum().unstack(fill_value=0)
display((dec.div(dec.sum(axis=1), axis=0) * 100).round(1).T)

decade,1970,1980,1990,2000,2010
region_txt,,,,,
Australasia & Oceania,0.2,0.3,0.4,0.1,0.1
Central America & Caribbean,9.3,24.0,6.4,0.2,0.0
Central Asia,0.0,0.0,1.2,0.6,0.1
East Asia,0.8,0.4,1.4,0.4,0.1
Eastern Europe,0.3,0.1,3.7,5.3,3.1
Middle East & North Africa,11.0,9.3,18.9,32.4,38.0
North America,16.1,1.8,2.3,1.1,0.4
South America,10.2,31.1,20.0,4.6,1.6
South Asia,0.5,9.2,16.8,32.1,33.6


In [13]:
from IPython.display import display

pivot = (data.groupby(["iyear", "region_txt"]).size()
             .unstack(fill_value=0))

peaks = (pivot.idxmax().rename("peak_year").to_frame()
              .join(pivot.max().rename("peak_incidents"))
              .sort_values("peak_year"))
display(peaks)

,peak_year,peak_incidents
region_txt,,
North America,1970,472
Western Europe,1979,1020
Central America & Caribbean,1981,1148
South America,1984,1492
Australasia & Oceania,1989,29
East Asia,1990,99
Central Asia,1992,77
Southeast Asia,2013,1188
Eastern Europe,2014,962


In [15]:
import numpy as np
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from IPython.display import display

# ---------- 1. Aggregate to country-year ----------
country_year = (data.groupby(["country_txt", "iyear"])
                    .agg(incidents=("eventid", "size"),
                         killed=("nkill", "sum"),
                         wounded=("nwound", "sum"))
                    .reset_index())
country_year["casualties"] = country_year["killed"] + country_year["wounded"]

# ---------- 2. Correlations: all data vs. without the 10 most extreme country-years ----------
def corr_summary(df):
    return pd.Series({
        "country-years": len(df),
        "Pearson": df["incidents"].corr(df["casualties"]),
        "Spearman": df["incidents"].corr(df["casualties"], method="spearman"),
        "Pearson (log1p)": np.log1p(df["incidents"]).corr(np.log1p(df["casualties"])),
    })

extreme = country_year.nlargest(10, "casualties").index
display(pd.DataFrame({
    "all data": corr_summary(country_year),
    "without top-10 casualty country-years": corr_summary(country_year.drop(extreme)),
}).round(3))

# ---------- 3. Log-log scatter with fitted line ----------
pos = country_year[country_year["casualties"] > 0].copy()
print(f"Dropped from the log plot (zero casualties): {len(country_year) - len(pos):,} country-years")

pos["log_inc"] = np.log10(pos["incidents"])
pos["log_cas"] = np.log10(pos["casualties"])
slope, intercept = np.polyfit(pos["log_inc"], pos["log_cas"], 1)
pos["residual"] = pos["log_cas"] - (slope * pos["log_inc"] + intercept)

fig = px.scatter(pos, x="incidents", y="casualties", log_x=True, log_y=True,
                 hover_data=["country_txt", "iyear"], opacity=0.5,
                 title="Incidents vs casualties by country-year (log-log)")
xs = np.linspace(pos["log_inc"].min(), pos["log_inc"].max(), 50)
fig.add_trace(go.Scatter(x=10**xs, y=10**(slope * xs + intercept), mode="lines",
                         name=f"fit (slope {slope:.2f})", line=dict(color="red")))
fig.update_layout(height=550)
fig.show()

# ---------- 4. Irregularities: distance from the fitted line ----------
cols = ["country_txt", "iyear", "incidents", "casualties"]
print("More casualties than the trend predicts:")
display(pos.nlargest(10, "residual")[cols])

print("Many incidents but few casualties (10+ incidents):")
display(pos[pos["incidents"] >= 10].nsmallest(10, "residual")[cols])

,all data,without top-10 casualty country-years
country-years,3762.000,3752.000
Pearson,0.851,0.814
Spearman,0.761,0.759
Pearson (log1p),0.817,0.812


Dropped from the log plot (zero casualties): 742 country-years


More casualties than the trend predicts:


,country_txt,iyear,incidents,casualties
1819,Kenya,1998,2,4226.0
3502,United States,2001,41,19523.0
2759,Saudi Arabia,1996,1,405.0
1040,Ethiopia,1984,1,400.0
1751,Japan,1995,20,6240.0
1879,Kuwait,2015,1,255.0
312,Belarus,2011,1,174.0
2391,Niger,1989,1,171.0
608,Chad,2008,8,1183.0
3289,Tunisia,1980,1,141.0


Many incidents but few casualties (10+ incidents):


,country_txt,iyear,incidents,casualties
3308,Turkey,1971,35,1.0
1599,Ireland,2014,33,1.0
1112,France,1997,130,4.0
1233,Greece,2000,28,1.0
371,Bolivia,1982,27,1.0
3468,United Kingdom,2014,103,4.0
1211,Greece,1977,47,2.0
2969,Spain,1999,47,2.0
1114,France,1999,46,2.0
378,Bolivia,1989,40,2.0


In [16]:
country_year["cas_per_incident"] = country_year["casualties"] / country_year["incidents"]
rho_lethality = country_year["incidents"].corr(country_year["cas_per_incident"], method="spearman")
print(f"Spearman, incidents vs casualties per incident: {rho_lethality:.2f}")

Spearman, incidents vs casualties per incident: 0.32


In [17]:
from IPython.display import display

hi = pos.nlargest(10, "residual")
lo = pos[pos["incidents"] >= 10].nsmallest(10, "residual")

for name, keys in [("High casualties, few incidents", hi),
                   ("Many incidents, few casualties", lo)]:
    sub = data.merge(keys[["country_txt", "iyear"]], on=["country_txt", "iyear"])
    print(f"\n=== {name}: {len(sub)} incidents ===")
    print(f"Missing nkill: {sub['nkill'].isna().mean():.0%} | "
          f"missing nwound: {sub['nwound'].isna().mean():.0%}")
    display(sub["attacktype1_txt"].value_counts(normalize=True).head(4).round(2))
    display(sub["targtype1_txt"].value_counts(normalize=True).head(4).round(2))
    display(sub["gname"].value_counts().head(3))

# Largest single incident in each high-casualty country-year
sub_hi = data.merge(hi[["country_txt", "iyear"]], on=["country_txt", "iyear"])
display(sub_hi.sort_values("casualties", ascending=False)
              .groupby(["country_txt", "iyear"]).head(1)
              [["country_txt", "iyear", "city", "gname", "attacktype1_txt", "nkill", "nwound"]])


=== High casualties, few incidents: 77 incidents ===
Missing nkill: 1% | missing nwound: 3%


attacktype1_txt
Facility/Infrastructure Attack    0.29
Unarmed Assault                   0.26
Bombing/Explosion                 0.21
Unknown                           0.06
Name: proportion, dtype: float64

targtype1_txt
Business                       0.23
Private Citizens & Property    0.17
Government (General)           0.16
Transportation                 0.13
Name: proportion, dtype: float64

gname
Unknown                         31
Earth Liberation Front (ELF)    13
Aum Shinri Kyo                   5
Name: count, dtype: int64


=== Many incidents, few casualties: 536 incidents ===
Missing nkill: 3% | missing nwound: 3%


attacktype1_txt
Bombing/Explosion                 0.73
Facility/Infrastructure Attack    0.16
Armed Assault                     0.08
Assassination                     0.01
Name: proportion, dtype: float64

targtype1_txt
Government (General)           0.26
Private Citizens & Property    0.21
Business                       0.19
Military                       0.07
Name: proportion, dtype: float64

gname
Unknown                                                 312
Corsican National Liberation Front- Historic Channel     68
Turkish People's Liberation Army                         23
Name: count, dtype: int64

,country_txt,iyear,city,gname,attacktype1_txt,nkill,nwound
47,United States,2001,New York City,Al-Qaida,Hijacking,1384.0,8190.0
8,Japan,1995,Tokyo,Aum Shinri Kyo,Unarmed Assault,13.0,5500.0
25,Kenya,1998,Nairobi,Al-Qaida,Bombing/Explosion,224.0,4000.0
67,Chad,2008,N'Djamena,Rebels,Armed Assault,160.0,1001.0
23,Saudi Arabia,1996,Dhahran,Saudi Hezbollah,Bombing/Explosion,19.0,386.0
1,Ethiopia,1984,Lalibela,Tigray Peoples Liberation Front (TPLF),Hostage Taking (Barricade Incident),200.0,200.0
76,Kuwait,2015,Kuwait,Najd Province of the Islamic State,Bombing/Explosion,28.0,227.0
75,Belarus,2011,Minsk,Unknown,Bombing/Explosion,13.0,161.0
2,Niger,1989,Goure district,Hezbollah,Bombing/Explosion,171.0,0.0
0,Tunisia,1980,Gafsa,Tunisian Armed Resistance,Bombing/Explosion,41.0,100.0


In [18]:
import numpy as np
import pandas as pd
from IPython.display import display

# ---------- 1. Country-year aggregation (rebuilt here so the cell is self-contained) ----------
country_year = (data.groupby(["country_txt", "iyear"])
                    .agg(incidents=("eventid", "size"),
                         killed=("nkill", "sum"),
                         wounded=("nwound", "sum"))
                    .reset_index())
country_year["casualties"] = country_year["killed"] + country_year["wounded"]

# ---------- 2. Correlation summary for any subset ----------
def corr_summary(df):
    return pd.Series({
        "country-years": len(df),
        "Pearson": df["incidents"].corr(df["casualties"]),
        "Spearman": df["incidents"].corr(df["casualties"], method="spearman"),
        "Pearson (log1p)": np.log1p(df["incidents"]).corr(np.log1p(df["casualties"])),
    })

# ---------- 3. All data vs. without the 10 country-years with the most casualties ----------
top10 = country_year.nlargest(10, "casualties")
display(top10[["country_txt", "iyear", "incidents", "casualties"]])   # which rows are removed

table = pd.DataFrame({
    "all data": corr_summary(country_year),
    "without top-10 country-years": corr_summary(country_year.drop(top10.index)),
}).round(3)
display(table)

,country_txt,iyear,incidents,casualties
1558,Iraq,2014,3933,30769.0
1560,Iraq,2016,3360,26495.0
1557,Iraq,2013,2852,22751.0
1559,Iraq,2015,2751,22211.0
1551,Iraq,2007,1047,19950.0
3502,United States,2001,41,19523.0
29,Afghanistan,2015,1928,13203.0
1550,Iraq,2006,838,12879.0
30,Afghanistan,2016,1617,12627.0
31,Afghanistan,2017,1414,12119.0


,all data,without top-10 country-years
country-years,3762.000,3752.000
Pearson,0.851,0.814
Spearman,0.761,0.759
Pearson (log1p),0.817,0.812


In [19]:
import pandas as pd
import plotly.express as px

overall = (data["attacktype1_txt"].value_counts()
               .rename_axis("attack_type").reset_index(name="incidents"))
overall["share_pct"] = (overall["incidents"] / overall["incidents"].sum() * 100).round(1)

fig = px.bar(overall.sort_values("incidents"), x="incidents", y="attack_type",
             orientation="h", text="share_pct",
             title="Most common attack methods (1970-2017)",
             labels={"incidents": "Incidents", "attack_type": ""})
fig.update_traces(texttemplate="%{text}%", textposition="outside")
fig.update_layout(height=450)
fig.show()

In [20]:
import pandas as pd
import plotly.express as px

region_order = data["region_txt"].value_counts().index.tolist()

# ---------- A. Attack-type mix by region (% of each region's incidents) ----------
mix = (pd.crosstab(data["region_txt"], data["attacktype1_txt"], normalize="index") * 100).round(1)
mix = mix.loc[region_order]

fig = px.imshow(mix, text_auto=".0f", aspect="auto", color_continuous_scale="Blues",
                title="Attack-type mix by region (% of the region's incidents)",
                labels={"x": "Attack type", "y": "Region", "color": "%"})
fig.update_layout(height=550)
fig.show()

# ---------- B. One panel per region: attack-type mix over time (5-year periods) ----------
top5 = [t for t in data["attacktype1_txt"].value_counts().index if t != "Unknown"][:5]
d = data.copy()
d["attack_group"] = d["attacktype1_txt"].where(d["attacktype1_txt"].isin(top5), "Other (incl. Unknown)")
d["period"] = (d["iyear"] // 5 * 5).astype(str)

t = (d.groupby(["region_txt", "period", "attack_group"]).size()
       .rename("incidents").reset_index())
t["share"] = t["incidents"] / t.groupby(["region_txt", "period"])["incidents"].transform("sum")

fig = px.bar(t, x="period", y="share", color="attack_group", facet_col="region_txt",
             facet_col_wrap=4, height=900,
             category_orders={"attack_group": top5 + ["Other (incl. Unknown)"],
                              "region_txt": region_order},
             color_discrete_sequence=px.colors.qualitative.Set2,
             title="Attack-type mix over time, by region (share of incidents per 5-year period)",
             labels={"period": "Period start", "share": "Share", "attack_group": "Attack type"})
fig.for_each_annotation(lambda a: a.update(text=a.text.split("=")[-1]))
fig.update_xaxes(tickangle=90, tickfont_size=9)
fig.show()

In [21]:
import pandas as pd
import plotly.express as px

# Top 5 methods + "Other" so the chart stays readable
top5 = [t for t in data["attacktype1_txt"].value_counts().index if t != "Unknown"][:5]
order = top5 + ["Other (incl. Unknown)"]

d = data.copy()
d["attack_group"] = d["attacktype1_txt"].where(d["attacktype1_txt"].isin(top5), "Other (incl. Unknown)")

yr = (d.groupby(["iyear", "attack_group"]).size()
        .rename("incidents").reset_index())
yr["share"] = yr["incidents"] / yr.groupby("iyear")["incidents"].transform("sum")

# ---------- A. Share of incidents by attack type, per year ----------
fig = px.bar(yr, x="iyear", y="share", color="attack_group",
             category_orders={"attack_group": order},
             color_discrete_sequence=px.colors.qualitative.Set2,
             title="Attack-type mix by year (share of incidents)",
             labels={"iyear": "Year", "share": "Share", "attack_group": "Attack type"})
fig.update_layout(height=500, bargap=0.1)
fig.show()

# ---------- B. Number of incidents by attack type, per year ----------
fig = px.area(yr, x="iyear", y="incidents", color="attack_group",
              category_orders={"attack_group": order},
              color_discrete_sequence=px.colors.qualitative.Set2,
              title="Incidents by attack type and year",
              labels={"iyear": "Year", "incidents": "Incidents", "attack_group": "Attack type"})
fig.update_layout(height=500)
fig.show()

In [22]:
share_year = (yr.pivot(index="iyear", columns="attack_group", values="share") * 100).round(1)
share_year["top_method"] = share_year[order].idxmax(axis=1)
display(share_year[share_year["top_method"] != "Bombing/Explosion"][["top_method"]])

attack_group,top_method
iyear,
1972,Assassination
1973,Assassination


In [24]:
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

# ---------- Coordinates: how complete are they? ----------
geo = data.dropna(subset=["latitude", "longitude"]).copy()
print(f"Incidents with coordinates: {len(geo):,} of {len(data):,} ({len(geo)/len(data):.1%})")
display((data["latitude"].isna().groupby(data["region_txt"]).mean() * 100)
        .round(1).rename("% missing coordinates").sort_values(ascending=False).to_frame())

# ---------- A. Random sample of incidents, colored by region ----------
sample = geo.sample(20000, random_state=42)

fig = px.scatter_geo(sample, lat="latitude", lon="longitude", color="region_txt",
                     hover_name="city", hover_data=["country_txt", "iyear", "attacktype1_txt", "nkill"],
                     opacity=0.5, projection="natural earth",
                     color_discrete_sequence=px.colors.qualitative.Dark24,
                     title="Terrorist incidents worldwide (random sample of 20,000, 1970-2017)",
                     labels={"region_txt": "Region"})
fig.update_traces(marker=dict(size=3))
fig.update_geos(showcountries=True, countrycolor="lightgray", showland=True, landcolor="#f2f2f2")
fig.update_layout(height=600, margin=dict(l=0, r=0, t=50, b=0))
fig.show()

# ---------- B. The 2,000 deadliest incidents, sized by casualties ----------
top = geo.dropna(subset=["casualties"]).nlargest(2000, "casualties")

fig = px.scatter_geo(top, lat="latitude", lon="longitude", size="casualties", color="region_txt",
                     hover_name="city", hover_data=["country_txt", "iyear", "gname", "nkill", "nwound"],
                     size_max=30, opacity=0.6, projection="natural earth",
                     color_discrete_sequence=px.colors.qualitative.Dark24,
                     title="The 2,000 incidents with the most casualties (bubble size = casualties)",
                     labels={"region_txt": "Region"})
fig.update_geos(showcountries=True, countrycolor="lightgray", showland=True, landcolor="#f2f2f2")
fig.update_layout(height=600, margin=dict(l=0, r=0, t=50, b=0))

fig.update_geos(projection_rotation=dict(lon=0))   # center the map on the Atlantic/Greenwich
fig.update_layout(height=650, legend=dict(orientation="h", y=-0.05))   # legend below, bigger map
fig.show()

Incidents with coordinates: 177,134 of 181,691 (97.5%)


,% missing coordinates
region_txt,
Central America & Caribbean,9.7
South America,4.6
Southeast Asia,4.3
Sub-Saharan Africa,3.1
Central Asia,2.7
Australasia & Oceania,2.1
Middle East & North Africa,1.8
South Asia,1.1
East Asia,1.0


In [25]:
import numpy as np
import pandas as pd
import plotly.express as px
from IPython.display import display

by_country = (data.groupby("country_txt")
                  .agg(incidents=("eventid", "size"), killed=("nkill", "sum"), wounded=("nwound", "sum"))
                  .reset_index())
by_country["casualties"] = by_country["killed"] + by_country["wounded"]

for metric, label in [("incidents", "Incidents"), ("casualties", "Casualties (killed + wounded)")]:
    by_country[f"log_{metric}"] = np.log10(by_country[metric].clip(lower=1))
    fig = px.choropleth(by_country, locations="country_txt", locationmode="country names",
                        color=f"log_{metric}", color_continuous_scale="Reds",
                        hover_name="country_txt", hover_data={metric: ":,", f"log_{metric}": False},
                        projection="natural earth",
                        title=f"{label} by country, 1970-2017 (log color scale)")
    fig.update_layout(height=550, margin=dict(l=0, r=0, t=50, b=0),
                      coloraxis_colorbar=dict(title=label.split(" ")[0], tickvals=[0, 1, 2, 3, 4, 5],
                                              ticktext=["1", "10", "100", "1k", "10k", "100k"]))
    fig.show()

# ---------- Top 10 countries ----------
display(by_country.nlargest(10, "incidents")[["country_txt", "incidents", "casualties"]])
display(by_country.nlargest(10, "casualties")[["country_txt", "incidents", "casualties"]])

# ---------- Countries that may not be shaded (defunct states / name mismatches) ----------
defunct = ["West Germany (FRG)", "East Germany (GDR)", "Soviet Union", "Yugoslavia", "Czechoslovakia",
           "South Yemen", "North Yemen", "Rhodesia", "Serbia-Montenegro", "South Vietnam",
           "People's Republic of the Congo", "Zaire", "International"]
share = data["country_txt"].isin(defunct).mean()
print(f"Incidents in defunct states or unmatched names: {share:.1%}")

,country_txt,incidents,casualties
84,Iraq,24636,213279.0
134,Pakistan,14368,65860.0
0,Afghanistan,12731,83661.0
80,India,11960,48321.0
35,Colombia,8306,25026.0
140,Philippines,6908,22926.0
139,Peru,6096,16849.0
52,El Salvador,5320,17115.0
188,United Kingdom,5235,9516.0
183,Turkey,4292,16787.0


,country_txt,incidents,casualties
84,Iraq,24636,213279.0
0,Afghanistan,12731,83661.0
134,Pakistan,14368,65860.0
80,India,11960,48321.0
130,Nigeria,3907,32921.0
167,Sri Lanka,3022,31091.0
175,Syria,2201,29338.0
35,Colombia,8306,25026.0
189,United States,2836,24473.0
140,Philippines,6908,22926.0


Incidents in defunct states or unmatched names: 0.6%


In [26]:
from IPython.display import display

# 1. Shares and lethality per country
total_inc = by_country["incidents"].sum()
by_country["share_of_incidents_pct"] = (by_country["incidents"] / total_inc * 100).round(1)
by_country["cas_per_incident"] = (by_country["casualties"] / by_country["incidents"]).round(1)

top_inc = by_country.nlargest(10, "incidents")
display(top_inc[["country_txt", "incidents", "share_of_incidents_pct", "casualties", "cas_per_incident"]])
print(f"Top 10 countries by incidents: {top_inc['incidents'].sum() / total_inc:.1%} of all incidents")

# 2. What is behind the UK's count?
display(data.loc[data["country_txt"] == "United Kingdom", "city"].value_counts().head(5))

# 3. Names Plotly may not match (these countries would appear unshaded)
suspects = ["West Bank and Gaza Strip", "Kosovo", "Ivory Coast", "Bosnia-Herzegovina", "Macedonia",
            "Republic of the Congo", "Democratic Republic of the Congo", "Hong Kong", "Macau",
            "Falkland Islands", "New Caledonia", "Guadeloupe", "Martinique", "French Guiana",
            "Western Sahara", "Serbia-Montenegro", "Zaire", "Yugoslavia", "Soviet Union",
            "West Germany (FRG)", "East Germany (GDR)", "Czechoslovakia", "South Yemen",
            "North Yemen", "Rhodesia", "South Vietnam", "International"]
unmatched = by_country[by_country["country_txt"].isin(suspects)].sort_values("incidents", ascending=False)
display(unmatched[["country_txt", "incidents"]])
print(f"Incidents under these names: {unmatched['incidents'].sum() / total_inc:.1%}")

,country_txt,incidents,share_of_incidents_pct,casualties,cas_per_incident
84,Iraq,24636,13.6,213279.0,8.7
134,Pakistan,14368,7.9,65860.0,4.6
0,Afghanistan,12731,7.0,83661.0,6.6
80,India,11960,6.6,48321.0,4.0
35,Colombia,8306,4.6,25026.0,3.0
140,Philippines,6908,3.8,22926.0,3.3
139,Peru,6096,3.4,16849.0,2.8
52,El Salvador,5320,2.9,17115.0,3.2
188,United Kingdom,5235,2.9,9516.0,1.8
183,Turkey,4292,2.4,16787.0,3.9


Top 10 countries by incidents: 55.0% of all incidents


city
Belfast        2170
London          399
Londonderry     366
Armagh          125
Newry           112
Name: count, dtype: int64

,country_txt,incidents
197,West Bank and Gaza Strip,2227
43,Democratic Republic of the Congo,775
198,West Germany (FRG),541
201,Yugoslavia,203
94,Kosovo,196
21,Bosnia-Herzegovina,159
106,Macedonia,118
145,Rhodesia,83
165,Soviet Union,78
88,Ivory Coast,74


Incidents under these names: 2.6%
